# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 0. 제출 정보와 분석 질문

이름: 장태원  
GitHub ID: pmsky1  
작성일: 2026-10-01  
사용한 AI 도구: ChatGPT

### 내가 확인하려는 질문

- 어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?
- `order_items`의 `order_id`, `product_id`, `quantity`, `unit_price`와 `orders`의 `order_status`, `order_date`, `customer_id`, `products`의 `category`를 사용함
- 주문상세 한 행의 `line_total = quantity × unit_price`를 계산한 뒤 `order_status == 'completed'`만 남김
- 카테고리별 `line_total` 합계와 `quantity` 합계를 집계하며 주문 건수는 고유 `order_id`로 계산함

### 나의 해석과 판단

- 취소·환불 주문이 섞이면 질문의 범위가 달라지므로 주문 상태를 확인하고 completed만 사용함
- 이 금액은 할인·배송비·세금·부분 환불을 반영한 회사의 실제 매출액이나 이익이 아님

![분석 질문과 계산 기준](images/step01_question.png)


## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        for project in [candidate, candidate / 'llm-data-analysis-course']:
            if (project / 'requirements.txt').exists() and (project / 'scripts').exists():
                return project
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('프로젝트 가상환경 사용:', Path(sys.executable).is_relative_to(PROJECT_ROOT / '.venv'))
print('현재 작업 폴더:', Path.cwd().name)
print('프로젝트 루트:', PROJECT_ROOT.name)
print('데이터 폴더:', DATA_DIR.relative_to(PROJECT_ROOT))
print('결과 저장 폴더:', REPORT_DIR.relative_to(PROJECT_ROOT))


프로젝트 가상환경 사용: True
현재 작업 폴더: chapter04
프로젝트 루트: llm-data-analysis-course
데이터 폴더: data/raw
결과 저장 폴더: reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [2]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [3]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


### 결과 관찰

- 고객 150행, 상품 100행, 주문 300행, 주문상세 764행과 필요한 컬럼을 확인함

### 나의 해석과 판단

- 카테고리는 products에, 주문 상태는 orders에 있어 두 파일을 주문상세와 연결해야 함

### 업무·분석적 의미

- 실제 컬럼과 파일 역할을 먼저 확인하면 없는 컬럼으로 코드를 작성하거나 집계 범위를 잘못 잡는 일을 줄일 수 있음

### 한계와 추가 확인 사항

- 파일 크기와 컬럼 확인만으로 값의 정확성이나 병합 가능성을 보장할 수 없어 이후 단계에서 직접 검증함


## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [4]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


## 5. 정렬과 주문 상태 확인


In [5]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False)[['customer_id', 'age', 'city']].head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts(dropna=False))


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
42,43,뷰티 상품 043,뷰티,197000
57,58,식품 상품 058,식품,197000
23,24,스포츠 상품 024,스포츠,196000
36,37,뷰티 상품 037,뷰티,193000
8,9,스포츠 상품 009,스포츠,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,age,city
78,79,69,부산
14,15,69,서울
8,9,69,서울
54,55,69,광주
136,137,68,울산
45,46,67,대전
132,133,67,성남
123,124,67,대구
57,58,66,성남
11,12,66,대전


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [6]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


### 필터링·정렬·파생 컬럼 결과

- 실제 도시 값과 주문 상태를 확인하고 고객 조건 필터와 상품 가격 내림차순 정렬을 실행함
- `line_total = quantity × unit_price`를 만들었고 전체 주문상세 금액은 255,610,000임

### 나의 해석과 판단

- 이 합계에는 cancelled와 refunded 주문도 포함될 수 있어 completed 기준 금액으로 해석할 수 없음

### 업무·분석적 의미

- 같은 계산식이라도 주문 상태 필터가 다르면 금액 지표의 의미가 달라짐

### 한계와 추가 확인 사항

- `unit_price`가 할인·쿠폰·배송비·세금을 어떻게 반영하는지는 현재 데이터만으로 확인할 수 없음

![필터링과 line_total 계산](images/step02_transform.png)


## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [7]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())
assert len(order_items) == len(order_sales)
assert order_sales['_merge'].eq('both').all()

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

### 주문 병합 결과 관찰

- `orders.order_id` 중복은 0건이며 주문상세 764행이 병합 뒤에도 764행임
- `validate='many_to_one'`을 통과했고 `_merge`는 both 764건, left_only 0건임

### 나의 해석과 판단

- 현재 데이터에서는 주문상세 각 행에 주문 상태가 한 번씩 연결되어 병합으로 금액이 중복될 위험을 줄였다고 판단함

### 업무·분석적 의미

- 오른쪽 키가 중복되어 행이 늘면 금액이 과대 집계될 수 있어 병합 직후 행 수와 매칭을 확인해야 함

### 한계와 추가 확인 사항

- 키가 맞아도 주문 상태나 단가 자체가 업무 기록과 일치하는지는 별도 확인이 필요함

![주문 데이터 병합 검증](images/step03_merge.png)


## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [8]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


### 완료 주문 범위 확인

- 주문 300건 중 completed는 184건이며 날짜 변환 실패는 0건임
- 완료 주문에 속한 주문상세는 474행이고 `line_total` 합계는 148,990,000임
- 전체 주문상세 금액 255,610,000과의 차이는 106,620,000임

### 나의 해석과 판단

- 질문의 범위인 completed 주문만 남긴 뒤 집계해야 함
- 주문상세 474행은 주문 184건과 다른 단위이므로 주문 수에는 `order_id.nunique()`를 사용함


## 9. 상품 데이터 병합과 검증


In [9]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())
assert len(completed_order_sales) == len(completed_sales_items)
assert completed_sales_items['_merge'].eq('both').all()

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

### 상품 병합 결과 관찰

- `products.product_id` 중복은 0건이며 완료 주문상세 474행이 상품 병합 뒤에도 474행임
- `_merge`는 both 474건, left_only 0건으로 확인함

### 나의 해석과 판단

- 현재 데이터에서는 카테고리 정보를 붙이는 과정에서 주문상세 행의 누락이나 증가가 없었음


## 10. 카테고리별·상품별 매출


In [10]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


## 11. 월별 매출


In [11]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-09,3529000,5,705800.0
1,2025-10,16197000,16,1012312.0
2,2025-11,11694000,17,687882.0
3,2025-12,12319000,14,879929.0
4,2026-01,5144000,8,643000.0
5,2026-02,10034000,12,836167.0
6,2026-03,14254000,18,791889.0
7,2026-04,16634000,20,831700.0
8,2026-05,14367000,19,756158.0
9,2026-06,8943000,9,993667.0


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [12]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


### 집계 결과 관찰

- 스포츠 카테고리가 31,743,000으로 가장 크고 완료 주문 기준 전체 금액의 21.31%임
- 상품별 최고 금액은 스포츠 상품 041의 5,705,000이며 월별 최고 금액은 2026-04의 16,634,000임
- 고객별 최고 금액은 익명 라벨 Customer 117의 4,100,000임

### 나의 해석과 판단

- 스포츠가 이번 샘플의 completed 주문 기준 금액 1위이므로 주문 수와 평균 주문 금액을 분리해 더 확인하고 싶음

### 업무·분석적 의미

- 카테고리와 월별 금액을 함께 보면 어떤 범위에서 변화가 생겼는지 후속 분석 질문을 정할 수 있음

### 한계와 추가 확인 사항

- 금액 1위가 가장 인기 있거나 이익이 가장 높은 카테고리라는 뜻은 아님
- 할인·환불·원가 정보가 없어 회계상 순매출이나 수익성을 판단할 수 없음

![카테고리와 상품 집계](images/step04_groupby.png)


## 12-1. 완료 주문 금액 교차 검증

같은 완료 주문 상세 행을 기준으로 만든 카테고리·상품·월·고객별 합계가 일치하는지 확인합니다.


In [13]:
assert completed_order_sales['order_date'].notna().all(), '완료 주문 날짜 누락'
assert completed_order_sales['customer_id'].notna().all(), '완료 주문 고객 ID 누락'
assert completed_sales_items['category'].notna().all(), '완료 주문 상품 카테고리 누락'

sales_totals = pd.Series({
    '완료 주문 상세': completed_order_sales['line_total'].sum(),
    '카테고리별': category_sales['total_sales'].sum(),
    '상품별': product_sales['total_sales'].sum(),
    '월별': monthly_summary['total_sales'].sum(),
    '고객별': customer_sales['total_sales'].sum(),
}, name='total_sales')
display(sales_totals.to_frame())
assert sales_totals.nunique() == 1, '집계 단위별 합계가 다릅니다'
print('총합 검증: PASS')


,total_sales
완료 주문 상세,148990000
카테고리별,148990000
상품별,148990000
월별,148990000
고객별,148990000


총합 검증: PASS


### 총합 일치 검증 결과

- 완료 주문상세·카테고리·상품·월·고객별 `total_sales` 합계가 모두 148,990,000으로 일치함
- 완료 주문 날짜와 고객 ID, 상품 카테고리에 결측이 없는지 확인한 뒤 총합 검증을 통과함

### 나의 해석과 판단

- 같은 주문 범위에서 집계 기준만 바꿨으므로 합계가 달라지면 해석 전에 필터, 병합, 결측, 날짜 변환 순서로 확인해야 함
- 총합 일치는 이번 계산의 일관성을 보여 주지만 원본 값 자체의 정확성을 증명하지는 않음

![집계별 총합 일치 검증](images/step05_total_check.png)


## 13. 결과 저장하기


In [14]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    saved = pd.read_csv(path)
    assert saved.shape == df.shape and saved.columns.tolist() == df.columns.tolist()
    assert saved['total_sales'].sum() == df['total_sales'].sum()
    print(filename, '재읽기 PASS', saved.shape, path.stat().st_size, 'bytes')


ch04_category_sales.csv 재읽기 PASS (7, 4) 246 bytes
ch04_product_sales.csv 재읽기 PASS (98, 5) 4310 bytes
ch04_monthly_sales.csv 재읽기 PASS (13, 4) 430 bytes
ch04_customer_sales.csv 재읽기 PASS (100, 5) 3286 bytes


### 저장 결과 관찰

- 카테고리·상품·월·고객별 CSV 4개를 저장하고 다시 읽어 행·열 구조와 금액 합계를 확인함
- 개인정보인 고객 이름은 분석 출력과 저장 파일에 포함하지 않음


## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


### 공식 예제 코드 검토 결과

- 검토 질문: 공식 예제 코드가 카테고리별 completed 주문 기준 금액을 계산하는지 ChatGPT와 함께 확인함
- 공식 예제 코드: `order_items.groupby('category')['line_total'].sum()`은 `order_items`에 category가 없어 바로 실행할 수 없음
- 검토 결과: 수량과 단가로 `line_total`을 만든 뒤 orders를 `order_id`로 다대일 병합해 completed만 남기고, products를 `product_id`로 다대일 병합한 후 category별로 합산해야 함
- 실제 검증: 위 실행 셀에서 orders·products 병합의 `validate`, `_merge`, 행 수와 다섯 가지 총합 148,990,000의 일치를 확인함
- 최종 판단: 공식 예제 코드는 그대로 사용하지 않고, 병합·필터·총합 검증을 포함한 별도 코드를 사용함

### 나의 해석과 판단

- 코드가 실행되어도 주문 상태 필터나 병합 검증이 빠지면 질문과 다른 금액을 만들 수 있으므로 실제 데이터와 총합으로 확인해야 함


### 월별 완료 주문 금액 코드 요청

~~~text
orders와 order_items를 사용해 completed 주문 기준 월별 금액을 계산하려고 합니다.

다음을 반드시 포함해 주세요.
- line_total = quantity × unit_price
- orders.order_id 고유성 확인
- many-to-one merge + indicator
- 병합 전후 행 수 확인
- 미매칭 확인
- 날짜 변환 실패 확인
- completed만 필터링
- 월별 total_sales와 고유 order_count 계산
- 결과를 회계상 순매출이라고 단정하지 않기
~~~

### ChatGPT 제안 코드와 실제 데이터 검증


In [1]:
from pathlib import Path
import pandas as pd

# 이 셀만 실행해도 로컬 수업 데이터를 찾을 수 있도록 경로를 확인함
def find_ch04_data_dir(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        options = [
            candidate / 'data' / 'raw',
            candidate / 'llm-data-analysis-course' / 'data' / 'raw',
        ]
        for data_dir in options:
            if all((data_dir / name).is_file() for name in ('orders.csv', 'order_items.csv')):
                return data_dir
    raise FileNotFoundError('수업 저장소의 data/raw 폴더를 찾지 못했습니다')

ch04_data_dir = find_ch04_data_dir(Path.cwd())
llm_orders = pd.read_csv(ch04_data_dir / 'orders.csv')
llm_items = pd.read_csv(ch04_data_dir / 'order_items.csv')

required_order_columns = {'order_id', 'order_date', 'order_status'}
required_item_columns = {'order_id', 'quantity', 'unit_price'}
assert required_order_columns.issubset(llm_orders.columns)
assert required_item_columns.issubset(llm_items.columns)
assert llm_orders['order_id'].notna().all(), 'orders.order_id 결측'
assert llm_orders['order_id'].is_unique, 'orders.order_id 중복'

llm_items = llm_items[['order_id', 'quantity', 'unit_price']].copy()
llm_items['line_total'] = llm_items['quantity'] * llm_items['unit_price']
llm_merged = llm_items.merge(
    llm_orders[['order_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)
print('병합 전/후 행 수:', len(llm_items), len(llm_merged))
print('병합 매칭:', llm_merged['_merge'].value_counts().to_dict())
assert len(llm_items) == len(llm_merged), '병합 후 행 수 변경'
assert llm_merged['_merge'].eq('both').all(), '주문에 없는 주문상세 발견'

llm_merged['order_date'] = pd.to_datetime(llm_merged['order_date'], errors='coerce')
date_failures = int(llm_merged['order_date'].isna().sum())
print('날짜 변환 실패:', date_failures)
assert date_failures == 0, '날짜 변환 실패를 확인한 뒤 집계를 진행하세요'

llm_completed = llm_merged.loc[
    llm_merged['order_status'].eq('completed')
].copy()
llm_completed['order_month'] = llm_completed['order_date'].dt.to_period('M').astype(str)
llm_monthly = (
    llm_completed.groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
    .reset_index(drop=True)
)
display(llm_monthly)
print('완료 주문상세 금액 합계:', int(llm_completed['line_total'].sum()))
print('월별 금액 합계:', int(llm_monthly['total_sales'].sum()))
assert llm_monthly['total_sales'].sum() == llm_completed['line_total'].sum()
assert llm_monthly['order_count'].sum() == llm_completed['order_id'].nunique()

# 앞에서 실행한 Notebook의 월별 결과가 남아 있으면 항목별로 비교함
if 'monthly_summary' in globals():
    existing = monthly_summary[['order_month', 'total_sales', 'order_count']].reset_index(drop=True)
    comparison = existing.merge(
        llm_monthly,
        on='order_month',
        how='outer',
        suffixes=('_notebook', '_llm'),
        indicator=True,
        validate='one_to_one',
    )
    display(comparison)
    assert comparison['_merge'].eq('both').all(), '월 목록 불일치'
    assert comparison['total_sales_notebook'].eq(comparison['total_sales_llm']).all(), '월별 금액 불일치'
    assert comparison['order_count_notebook'].eq(comparison['order_count_llm']).all(), '월별 주문 수 불일치'
    print('기존 Notebook 월별 집계와 일치: PASS')

print('이 금액은 quantity × unit_price의 완료 주문 합계이며 회계상 순매출이 아닙니다')


병합 전/후 행 수: 764 764
병합 매칭: {'both': 764, 'left_only': 0, 'right_only': 0}
날짜 변환 실패: 0


,order_month,total_sales,order_count
0,2025-09,3529000,5
1,2025-10,16197000,16
2,2025-11,11694000,17
3,2025-12,12319000,14
4,2026-01,5144000,8
5,2026-02,10034000,12
6,2026-03,14254000,18
7,2026-04,16634000,20
8,2026-05,14367000,19
9,2026-06,8943000,9


완료 주문상세 금액 합계: 148990000
월별 금액 합계: 148990000
이 금액은 quantity × unit_price의 완료 주문 합계이며 회계상 순매출이 아닙니다


### 월별 코드 검증 결과

- LLM Prompt 요약: 완료 주문 기준 월별 금액을 계산하되 키 고유성, 다대일 병합, 미매칭·행 수·날짜 검증, 고유 주문 수와 금액의 의미를 확인하도록 요청함
- 제안 코드 요약: `line_total`을 만든 뒤 주문과 병합하고 `completed` 주문만 남겨 `order_month`별 금액 합계와 고유 주문 수를 계산함
- 실제 컬럼과 연결 키: `orders`의 `order_id`, `order_date`, `order_status`와 `order_items`의 `order_id`, `quantity`, `unit_price`를 확인하고 `order_id`로 병합함. 주문상세 금액은 `quantity × unit_price`로 계산함
- 키와 병합 검증: `orders.order_id`의 결측·중복 검사 및 `validate='many_to_one'`을 통과함. 주문상세 764행이 병합 후에도 764행이고 764행 모두 매칭되어 미매칭은 0행임
- 날짜와 주문 범위: 날짜 변환 실패는 0건이며 `order_status == 'completed'`인 주문상세만 월별로 집계함
- 월별 결과: 2025-09부터 2026-09까지 13개월, `total_sales` 합계 148,990,000원, 월별 고유 `order_count` 합계 184건임. 완료 주문상세 금액 합계도 148,990,000원으로 일치함
- 기존 집계 비교: 새 실행에서는 `monthly_summary` 변수가 없어 코드의 선택적 비교 출력은 나타나지 않았음. 대신 이 Notebook에 저장된 앞선 월별 집계 Output과 새 Output의 13개 월별 금액·주문 수를 대조해 모두 일치함을 확인함
- 실제 컬럼·범위와 맞지 않거나 빠진 검증 항목: 이번 월별 제안 코드에서는 발견하지 못함. 앞선 공식 카테고리 예제에는 `order_items.category`가 없고 완료 주문 필터가 빠져 있어 별도로 수정했음
- 최종 판단: 제안 코드는 실제 컬럼과 연결 관계에 맞고 행 수·미매칭·날짜·총합 검증을 통과해 완료 주문 기준 월별 분석에 사용함. 코드가 실행된다는 사실만으로는 필터·집계 단위의 정확성을 보장하지 않으므로 실제 Output과 기존 집계도 대조했음. 이 금액은 할인·배송비·세금·환불을 반영한 회계상 순매출로 해석하지 않음

![월별 완료 주문 코드 검증 실행 결과](images/step06_llm_validation.png)


## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [15]:
customers_over_40 = customers.loc[customers['age'] >= 40, ['customer_id', 'age', 'city']]
print('40세 이상 고객 수:', len(customers_over_40))
display(customers_over_40.head())

display(products.sort_values('price')[['product_id', 'product_name', 'category', 'price']].head(10))

payment_summary = orders['payment_method'].value_counts(dropna=False).rename_axis('payment_method').reset_index(name='order_count')
payment_summary['ratio_pct'] = (payment_summary['order_count'] / len(orders) * 100).round(2)
display(payment_summary)

display(products.groupby('category', as_index=False).agg(average_price=('price', 'mean')).sort_values('average_price', ascending=False))

print('전체 주문 상세 금액:', all_order_amount)
print('완료 주문 기준 금액:', sales_totals['완료 주문 상세'])
print('전체와 완료 주문의 차이:', all_order_amount - sales_totals['완료 주문 상세'])


40세 이상 고객 수: 78


,customer_id,age,city
2,3,61,성남
3,4,55,울산
6,7,53,인천
8,9,69,서울
9,10,62,울산


,product_id,product_name,category,price
27,28,뷰티 상품 028,뷰티,5000
5,6,전자기기 상품 006,전자기기,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000


,payment_method,order_count,ratio_pct
0,kakao_pay,79,26.33
1,naver_pay,77,25.67
2,bank_transfer,74,24.67
3,card,70,23.33


,category,average_price
4,식품,137142.857143
1,뷰티,117687.500000
6,패션,115909.090909
3,스포츠,111578.947368
0,도서,106857.142857
5,전자기기,101588.235294
2,생활용품,96437.500000


전체 주문 상세 금액: 255610000
완료 주문 기준 금액: 148990000
전체와 완료 주문의 차이: 106620000


### 고객별 금액 코드 요청 Prompt

```text
현재 데이터의 orders(order_id, customer_id, order_status)와 order_items(order_id, quantity, unit_price)를 사용해 completed 주문 기준 고객별 금액을 계산하는 pandas 코드를 제안해 주세요.
line_total = quantity × unit_price를 사용하고 orders.order_id 고유성, many_to_one 병합, indicator의 미매칭, 병합 전후 행 수를 확인해 주세요.
고객별 주문 수는 order_id.nunique()로 계산하고 고객 이름은 출력하지 말아 주세요. 고객별 합계와 완료 주문상세 합계가 일치하는지도 검증해 주세요.
```

## Chapter 04 최종 인사이트

### 가장 의미 있다고 생각한 결과 2가지

- 스포츠 카테고리가 완료 주문 기준 31,743,000으로 가장 높고 전체의 21.31%를 차지함
- 전체 주문상세 금액과 완료 주문 금액의 차이가 106,620,000이어서 주문 상태 범위를 명시해야 함

### 추가로 확인하고 싶은 질문

- 스포츠 카테고리 금액 1위가 주문 건수, 판매 수량, 높은 단가 중 무엇의 영향을 많이 받았는지 확인하고 싶음

### 현재 결과의 한계

- 샘플 데이터의 `quantity × unit_price` 합계만으로 실제 순매출, 이익, 인기 원인을 알 수 없음


## 최종 제출 체크

- [x] 처음부터 끝까지 셀을 실행하고 핵심 Output을 남김
- [x] 두 병합의 행 수와 미매칭, 다섯 가지 총합을 검증함
- [x] 결과 관찰과 나의 해석, 업무 의미와 한계를 구분함
- [x] 공식 예제 코드를 실제 컬럼과 완료 주문 범위로 검토함
- [x] 고객 이름과 Secret을 출력·기록하지 않음
- [x] GitHub에서 Notebook 표시를 확인함
- [x] 최종 Notebook 파일 URL을 LMS에 제출함


## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.
